# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [15]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [16]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [17]:
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'

df         = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END   = 4000
VAL_START   = 4000
VAL_END     = 5000

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds   = hf_dataset.select(range(VAL_START, VAL_END))

display(df.head())

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [18]:
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'

tokenizer           = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [19]:
TEXT_COLUMN       = 'sms'
PADDING_STRATEGY  = 'max_length'
TRUNCATION_FLAG   = True
MAX_SEQ_LEN       = 64

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [20]:
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [21]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5  # 1/√d_k

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(
            query,
            key.transpose(-2, -1),  # transpose la dernière dimension
        ) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)  # dernière dimension

        return torch.matmul(attn, value), attn  # poids × valeurs


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn      = Attention(embed_dim)
        self.fc        = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed                = self.embedding(x)
        attn_output, _       = self.attn(embed, embed, embed)  # self-attention q=k=v
        pooled               = attn_output.mean(dim=1)         # moyenne sur les tokens
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [22]:
ATTN_TEXT_COLUMN = 'sms'
ATTN_MAX_LEN     = 64

def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn   = val_ds.map(preprocess_for_attention)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [23]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label':     torch.tensor(item['label'],     dtype=torch.long),
        }

TRAIN_DATA_FOR_LOADER = train_ds_attn
VAL_DATA_FOR_LOADER   = val_ds_attn

train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader   = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER),   batch_size=32)

In [24]:
vocab_size    = len(tokenizer)  # taille vocabulaire GPT-2 avec tokens ajoutés
embed_dim     = 64
num_classes   = 2
learning_rate = 1e-3

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer  = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion  = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss    = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

Custom Attention model trained on SMS dataset. Sample batch loss: 0.2165900468826294


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [25]:
!pip install --quiet evaluate

import evaluate
import numpy as np

accuracy  = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall    = evaluate.load('recall')
f1        = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy.compute( predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall':    recall.compute(   predictions=preds, references=labels)['recall'],
        'f1':        f1.compute(       predictions=preds, references=labels)['f1'],
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [26]:
gpt2_preds, gpt2_labels = [], []
model.eval()

for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    'accuracy':  accuracy.compute( predictions=gpt2_preds,  references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds,  references=gpt2_labels)['precision'],
    'recall':    recall.compute(   predictions=gpt2_preds,  references=gpt2_labels)['recall'],
    'f1':        f1.compute(       predictions=gpt2_preds,  references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


GPT-2 Metrics: {'accuracy': 0.142, 'precision': 0.13941825476429287, 'recall': 1.0, 'f1': 0.24471830985915494}


In [27]:
attn_preds, attn_labels = [], []
attn_model.eval()

for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds   = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    'accuracy':  accuracy.compute( predictions=attn_preds,  references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds,  references=attn_labels)['precision'],
    'recall':    recall.compute(   predictions=attn_preds,  references=attn_labels)['recall'],
    'f1':        f1.compute(       predictions=attn_preds,  references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)

Attention Model Metrics: {'accuracy': 0.854, 'precision': 0.37037037037037035, 'recall': 0.07194244604316546, 'f1': 0.12048192771084337}


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
### 1. Rôles de Query, Key et Value

- **Query (Q)** : représente le mot actuel qui "cherche" des informations
  pertinentes dans la séquence. C'est la question posée.
- **Key (K)** : représente ce que chaque mot "offre" comme information.
  Le score entre Q et K mesure la pertinence.
- **Value (V)** : contient l'information réelle qu'on récupère.
  Plus le score Q·K est élevé, plus on prend de V.

Exemple : dans "Tu as gagné 500 000 FCFA, appelle maintenant",
"gagné" (Q) a un score élevé avec "FCFA" (K), donc on récupère
beaucoup de l'information de "FCFA" (V) — signal fort de spam.


### 2. Why do we use a scaling factor in the dot-product attention?
### 2. Pourquoi le facteur d'échelle 1/√d_k ?

Quand embed_dim est grand, les produits scalaires Q·K deviennent
très grands. Appliqué au softmax, cela produit des probabilités
proches de 0 ou 1, ce qui bloque les gradients et empêche
l'apprentissage. Diviser par √d_k ramène les scores dans une
plage normale où le softmax reste utile.

### 3. How does self-attention differ from traditional sequence models like RNNs?

### 3. Self-attention vs RNN

| Critère | RNN | Self-Attention |
|---------|-----|----------------|
| Traitement | Séquentiel (mot par mot) | Parallèle (tous les mots) |
| Dépendances longues | Difficile (oubli progressif) | Direct (accès immédiat) |
| Vitesse d'entraînement | Lent | Rapide |
| Mémoire | Limitée par l'état caché | Limitée par max_length |

Le RNN traite "Je mange du attiéké" mot par mot et peut oublier
"Je" avant d'arriver à "attiéké". La self-attention voit tous
les mots simultanément et peut relier "Je" à "attiéké" directement.


### 4. Performance analysis
### 4. Analyse des performances

GPT-2 obtiendra probablement de meilleures métriques car il a été
pré-entraîné sur 40 Go de textes — ses représentations sont riches
dès le départ, même sans fine-tuning spécifique au spam.

Le modèle custom est plus léger et s'entraîne en quelques secondes,
mais il part de zéro avec embed_dim=64 seulement.

Pour améliorer le modèle custom : augmenter embed_dim (128 ou 256),
ajouter plusieurs couches d'attention (multi-head), ou entraîner
sur plusieurs epochs au lieu d'une seule.
